# Mind2Web Debug Evaluation

Quick single-sample debug notebook for testing Magma inference on Mind2Web.

Loads the model, runs inference on one sample, and compares the output to ground truth.

## 1. Setup and Imports

In [1]:
# Install required dependencies (run this on Colab or if packages are missing)
# Magma requires transformers 4.49.0 - 4.51.x (NOT 5.x which has breaking changes)

%pip install open_clip_torch peft bitsandbytes accelerate -q
# %pip install "transformers>=4.49.0,<5.0.0" -q  # Pin to 4.x to avoid breaking changes
%pip install git+https://github.com/jwyang/transformers.git@dev/jwyang-v4.48.2

Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/jwyang/transformers.git (to revision dev/jwyang-v4.48.2) to /tmp/pip-req-build-uje3n5y4
  Running command git clone --filter=blob:none --quiet https://github.com/jwyang/transformers.git /tmp/pip-req-build-uje3n5y4
  Running command git checkout -b dev/jwyang-v4.48.2 --track origin/dev/jwyang-v4.48.2
  Switched to a new branch 'dev/jwyang-v4.48.2'
  Branch 'dev/jwyang-v4.48.2' set up to track remote branch 'dev/jwyang-v4.48.2' from 'origin'.
  Resolved https://github.com/jwyang/transformers.git to commit 1708d3ca3608b3c229d7ce6f0351e652436d7d33
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Check transformers version after install
import transformers
print(f"Transformers version: {transformers.__version__}")

from packaging import version
tf_version = version.parse(transformers.__version__)

if tf_version < version.parse("4.49.0"):
    print("Warning: Transformers version is too old! Magma requires >=4.49.0")
    print("Please restart the kernel for the new transformers version to take effect!")
elif tf_version >= version.parse("5.0.0"):
    print("Warning: Transformers 5.x detected - this has breaking changes with Magma!")
    print("Please run: %pip install 'transformers>=4.49.0,<5.0.0' -q")
    print("Then restart the kernel!")
else:
    print("Transformers version is compatible with Magma")

/home/thaole/miniconda3/envs/magma/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/thaole/miniconda3/envs/magma/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 4.48.2
Please restart the kernel for the new transformers version to take effect!


In [3]:
import os
import sys
import json
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
from IPython.display import display

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4070 Ti SUPER


In [4]:
# Patch for PyTorch 2.10+ compatibility with Magma model
# The model uses sum() on boolean tensors which requires explicit conversion in newer PyTorch

if not hasattr(torch, '_original_sum_backup'):
    torch._original_sum_backup = torch.sum

def _patched_sum(input, *args, **kwargs):
    """Patched sum that converts bool tensors to long before summing with dim."""
    if isinstance(input, bool):
        input = torch.tensor(input, dtype=torch.long)
    elif isinstance(input, torch.Tensor) and input.dtype == torch.bool and (len(args) > 0 or 'dim' in kwargs):
        input = input.long()
    return torch._original_sum_backup(input, *args, **kwargs)

torch.sum = _patched_sum
print("Applied PyTorch sum() patch for boolean tensor compatibility")

Applied PyTorch sum() patch for boolean tensor compatibility


## 2. Configuration

In [ ]:
# Base model name
BASE_MODEL = "microsoft/Magma-8B"

# # Dataset config
# HF_DATASET_NAME = "MagmaAI/Magma-Mind2Web-SoM"
# HF_DATASET_SPLIT = "train"

# # Which sample index to debug
# SAMPLE_INDEX = 0
IMAGE_PATH = "/home/thaole/thao_le/Magma/evaluation//ScreenSpot 2.png"
INSTRUCTION = "adjust the style"                             # từ ScreenSpot annotation
GROUND_TRUTH_BBOX = [0.6614583333333334,
  0.6666666666666666,
  0.7052083333333333,
  0.7833333333333333]   # từ ScreenSpot bbox column

image = Image.open(IMAGE_PATH).convert('RGB')
print(f"Base model: {BASE_MODEL}")
print(f"Image size: {image.size}")
display(image)
# print(f"Dataset: {HF_DATASET_NAME} (split: {HF_DATASET_SPLIT})")
# print(f"Sample index: {SAMPLE_INDEX}")


FileNotFoundError: [Errno 2] No such file or directory: '/content/ScreenSpot 2.png'

## 3. Load Dataset

In [ ]:
# from datasets import load_dataset

# print(f"Loading dataset: {HF_DATASET_NAME} (split: {HF_DATASET_SPLIT})")
# dataset = load_dataset(HF_DATASET_NAME, split=HF_DATASET_SPLIT)
# print(f"Dataset loaded! Size: {len(dataset)}")
# print(f"Columns: {dataset.column_names}")

## 4. Load Model

In [ ]:
print(f"Loading model: {BASE_MODEL}")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

print(f"Model config mm_use_image_start_end: {model.config.mm_use_image_start_end}")
print("Model loaded!")

## 5. Inspect Sample

In [ ]:
# Get sample from dataset
# sample = dataset[SAMPLE_INDEX]
# print("Conversations:", sample['conversations'])
# print("\nUser prompt:", sample['conversations'][0]['value'])
# print("\nGround truth:", sample['conversations'][1]['value'])

# image = sample['image'].convert('RGB')
# full_prompt = sample['conversations'][0]['value']  # Keep <image> tag
full_prompt = f"<image_start><image><image_end>\n{INSTRUCTION}"
# Display the image
print(f"\n{'='*60}")
print("INPUT IMAGE:")
print('='*60)
display(image)

print(f"\n{'='*60}")
print("INPUT PROMPT (from dataset):")
print('='*60)
print(full_prompt)

## 6. Run Inference

In [ ]:
# Always use Magma's correct image token format
# full_prompt = full_prompt.replace('<image>', '<image_start><image><image_end>')

# Build conversation
convs = [
    {"role": "system", "content": "You are agent that can see, talk and act."},
    {"role": "user", "content": full_prompt},
]

formatted_prompt = processor.tokenizer.apply_chat_template(
    convs, tokenize=False, add_generation_prompt=True
)

# Run inference
inputs = processor(images=[image], texts=formatted_prompt, return_tensors="pt")
inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
inputs['image_sizes'] = inputs['image_sizes'].unsqueeze(0)
# Move to device, cast only pixel_values to bfloat16 (input_ids must stay Long)
inputs = inputs.to('cuda')
inputs['pixel_values'] = inputs['pixel_values'].to(torch.bfloat16)

model.generation_config.pad_token_id = processor.tokenizer.pad_token_id

print("Running inference...")
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        temperature=0.0,
        do_sample=False,
        num_beams=1,
        max_new_tokens=256,
        use_cache=False
    )

prompt_decoded = processor.batch_decode(inputs['input_ids'], skip_special_tokens=True)[0]
response = processor.batch_decode(output_ids, skip_special_tokens=True)[0]
response = response.replace(prompt_decoded, '').strip()

print(f"\n{'='*60}")
print("MODEL RESPONSE:")
print('='*60)
print(response)

print(f"\n{'='*60}")
print("EXPECTED OUTPUT:", GROUND_TRUTH_BBOX)
print('='*60)

print(f"\n{'='*60}")
print("EVALUATION:")
print('='*60)

import re

def parse_coordinate(text):
    match = re.search(r'Coordinate:\s*\(([\d.]+),\s*([\d.]+)\)', text)
    if match:
        return float(match.group(1)), float(match.group(2))
    match = re.search(r'\(([\d.]+),\s*([\d.]+)\)', text)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None

def check_click_accuracy(pred_x, pred_y, bbox):
    """Check if predicted (x,y) falls inside ground truth bbox [x1,y1,x2,y2]."""
    x1, y1, x2, y2 = bbox
    return x1 <= pred_x <= x2 and y1 <= pred_y <= y2

# expected_raw = sample['conversations'][1]['value']
coord = parse_coordinate(response)
if coord:
    pred_x, pred_y = coord
    is_correct = check_click_accuracy(pred_x, pred_y, GROUND_TRUTH_BBOX)
# try:
#     expected_parsed = json.loads(expected_raw) if isinstance(expected_raw, str) else expected_raw
#     print(json.dumps(expected_parsed, indent=2))
# except json.JSONDecodeError:
#     print(expected_raw)

## 7. Cleanup

In [ ]:
del model
torch.cuda.empty_cache()
print("Model deleted, GPU memory freed.")